In [1]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, BaseMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing import TypedDict, Annotated

C:\Users\YASH\AppData\Local\Temp\ipykernel_28904\2152959881.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [3]:
# Load the document
loader = PyPDFLoader("intro-to-ml.pdf")
docs = loader.load()

In [4]:
len(docs)

392

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [6]:
len(chunks)

973

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store

In [ ]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 4})

In [ ]:
retriever.invoke("What is Decision Tree")

In [ ]:
@tool
def rag_tool(query):
    """
    Retrieve relevant information from the pdf document.
    Use this tool when the user ask factual / conceptual questions
    that might be answered from the stored documents.
    """
    
    result = retriever.invoke(query)
    
    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]
    
    return {
        'query': query,
        'context': context,
        'metadata': metadata
    }

In [ ]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

### LangGraph Portion

In [ ]:
class ChateState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    

In [ ]:
def chat_node(state: ChateState):
    messages = state['messages']
    
    response = llm_with_tools.invoke(messages)
    
    # append response(s) to the existing messages list (avoid creating a set)
    if isinstance(response, list):
        new_messages = messages + response
    else:
        new_messages = messages + [response]
    
    return {'messages': new_messages}

In [ ]:
tool_node = ToolNode(tools)

In [ ]:
graph = StateGraph(ChateState)

graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')

chatbot = graph.compile()


In [ ]:
chatbot

In [ ]:
result = chatbot.invoke({
    'messages': [HumanMessage(content="Using the pdf nodets, explain how to find the ideal value of k in KNN")]
})

In [ ]:
ai_response = result['messages'][-1].content
print(ai_response)